In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# On Kaggle, input data lives somewhere under /kaggle/input (read-only), nested
# under a competition-specific subfolder. Locally (e.g. VS Code), it's flat in ./data,
# populated by `kaggle competitions download -c titanic`. Either way, find the folder
# that actually contains train.csv rather than hardcoding the path.

import os

SEARCH_ROOT = '/kaggle/input' if os.path.isdir('/kaggle/input') else 'data'
WORKING_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else 'output'

INPUT_DIR = None
for dirname, _, filenames in os.walk(SEARCH_ROOT):
    for filename in filenames:
        print(os.path.join(dirname, filename))
    if 'train.csv' in filenames:
        INPUT_DIR = dirname

if INPUT_DIR is None:
    raise FileNotFoundError(f'Could not find train.csv under {SEARCH_ROOT}')

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

data/test.csv
data/train.csv
data/gender_submission.csv


## Load data

In [2]:
train = pd.read_csv(os.path.join(INPUT_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(INPUT_DIR, 'test.csv'))

print(train.shape, test.shape)
train.head()

(891, 12) (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Feature engineering

- `Title` extracted from `Name` (Mr/Mrs/Miss/Master/Rare) tends to capture age/sex/social status signal better than `Name` itself.
- `FamilySize` = `SibSp` + `Parch` + 1, and `IsAlone` as a flag, since family size affected survival odds.
- Missing `Age`/`Fare`/`Embarked` are imputed inside the model pipeline so train/test are handled identically.

In [3]:
def engineer_features(df):
    df = df.copy()

    df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.')
    title_map = {
        'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
        'Lady': 'Rare', 'Countess': 'Rare', 'Capt': 'Rare', 'Col': 'Rare',
        'Don': 'Rare', 'Dr': 'Rare', 'Major': 'Rare', 'Rev': 'Rare',
        'Sir': 'Rare', 'Jonkheer': 'Rare', 'Dona': 'Rare',
    }
    df['Title'] = df['Title'].replace(title_map)
    df.loc[~df['Title'].isin(['Mr', 'Mrs', 'Miss', 'Master', 'Rare']), 'Title'] = 'Rare'

    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

    # First letter of Cabin as deck; 'U' (unknown) for the ~77% of passengers with no cabin recorded.
    df['Deck'] = df['Cabin'].str[0].fillna('U')

    return df

train_fe = engineer_features(train)
test_fe = engineer_features(test)

train_fe[['Name', 'Title', 'FamilySize', 'IsAlone', 'Deck']].head()

,Name,Title,FamilySize,IsAlone,Deck
0,"Braund, Mr. Owen Harris",Mr,2,0,U
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs,2,0,C
2,"Heikkinen, Miss. Laina",Miss,1,1,U
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs,2,0,C
4,"Allen, Mr. William Henry",Mr,1,1,U


In [4]:
# Passengers sharing a Ticket number traveled as a group (often a family), and Fare on
# shared tickets is the total for the group, not per-person. Both are known from the
# ticket/fare fields alone, so computing them over train+test together isn't label leakage.
ticket_counts = pd.concat([train_fe['Ticket'], test_fe['Ticket']]).value_counts()

for df in (train_fe, test_fe):
    df['TicketGroupSize'] = df['Ticket'].map(ticket_counts)
    df['FarePerPerson'] = df['Fare'] / df['TicketGroupSize']

train_fe[['Ticket', 'Fare', 'TicketGroupSize', 'FarePerPerson']].head()

,Ticket,Fare,TicketGroupSize,FarePerPerson
0,A/5 21171,7.2500,1,7.25000
1,PC 17599,71.2833,2,35.64165
2,STON/O2. 3101282,7.9250,1,7.92500
3,113803,53.1000,2,26.55000
4,373450,8.0500,1,8.05000


## Build preprocessing + model pipeline

Numeric features are median-imputed and scaled; categorical features are mode-imputed and one-hot encoded. Wrapping this in a `Pipeline` means the same transforms are learned on train and applied to test without leakage.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV
from xgboost import XGBClassifier

numeric_features = ['Age', 'Fare', 'FamilySize', 'IsAlone', 'TicketGroupSize', 'FarePerPerson']
categorical_features = ['Pclass', 'Sex', 'Embarked', 'Title', 'Deck']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

X = train_fe[numeric_features + categorical_features]
y = train_fe['Survived']
X_test = test_fe[numeric_features + categorical_features]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Compare a few models with cross-validation

5-fold stratified cross-validation accuracy gives a more honest estimate than a single train/test split, since the dataset is small (~890 rows).

In [6]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'RandomForest': RandomForestClassifier(n_estimators=300, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(random_state=42),
    'XGBoost': XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42, eval_metric='logloss'),
}

results = {}
for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    scores = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy')
    results[name] = scores
    print(f'{name:18s} mean={scores.mean():.4f}  std={scores.std():.4f}')

LogisticRegression mean=0.8260  std=0.0160


RandomForest       mean=0.8170  std=0.0240


GradientBoosting   mean=0.8406  std=0.0164


XGBoost            mean=0.8417  std=0.0207


## Hyperparameter tuning

The two tree-boosting models score highest above, so grid-search their key hyperparameters (number/depth of trees, learning rate) with the same 5-fold CV to see if either beats the untuned baseline.

In [7]:
param_grid_gb = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [2, 3, 4],
    'model__learning_rate': [0.03, 0.05, 0.1],
}
gb_pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', GradientBoostingClassifier(random_state=42))])
gb_search = GridSearchCV(gb_pipe, param_grid_gb, cv=cv, scoring='accuracy', n_jobs=-1)
gb_search.fit(X, y)
print('GradientBoosting best params:', gb_search.best_params_)
print(f'GradientBoosting best CV accuracy: {gb_search.best_score_:.4f}')

GradientBoosting best params: {'model__learning_rate': 0.03, 'model__max_depth': 4, 'model__n_estimators': 100}
GradientBoosting best CV accuracy: 0.8518


In [8]:
param_grid_xgb = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [2, 3, 4],
    'model__learning_rate': [0.03, 0.05, 0.1],
    'model__subsample': [0.8, 1.0],
}
xgb_pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', XGBClassifier(random_state=42, eval_metric='logloss'))])
xgb_search = GridSearchCV(xgb_pipe, param_grid_xgb, cv=cv, scoring='accuracy', n_jobs=-1)
xgb_search.fit(X, y)
print('XGBoost best params:', xgb_search.best_params_)
print(f'XGBoost best CV accuracy: {xgb_search.best_score_:.4f}')

XGBoost best params: {'model__learning_rate': 0.05, 'model__max_depth': 4, 'model__n_estimators': 200, 'model__subsample': 0.8}
XGBoost best CV accuracy: 0.8530


## Train final model and write submission

In [9]:
candidates = {
    name: (scores.mean(), Pipeline(steps=[('preprocessor', preprocessor), ('model', models[name])]))
    for name, scores in results.items()
}
candidates['GradientBoosting (tuned)'] = (gb_search.best_score_, gb_search.best_estimator_)
candidates['XGBoost (tuned)'] = (xgb_search.best_score_, xgb_search.best_estimator_)

for name, (score, _) in sorted(candidates.items(), key=lambda kv: -kv[1][0]):
    print(f'{name:28s} {score:.4f}')

best_name = max(candidates, key=lambda name: candidates[name][0])
best_score, final_pipe = candidates[best_name]
print(f'\nBest: {best_name} (cv={best_score:.4f})')

final_pipe.fit(X, y)
predictions = final_pipe.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test_fe['PassengerId'],
    'Survived': predictions,
})
os.makedirs(WORKING_DIR, exist_ok=True)
submission.to_csv(os.path.join(WORKING_DIR, 'submission.csv'), index=False)
submission.head()

XGBoost (tuned)              0.8530
GradientBoosting (tuned)     0.8518
XGBoost                      0.8417
GradientBoosting             0.8406
LogisticRegression           0.8260
RandomForest                 0.8170

Best: XGBoost (tuned) (cv=0.8530)


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
